In [ ]:
import os
import pandas as pd

# ============================================================
# Lab 03 — Data cleaning (portable, deterministic)
# Chạy từ REPO ROOT. Input file gốc -> data/asr.csv (canonical input cả nhóm).
# ============================================================

INPUT_FILE  = os.path.join("Lab03", "Amazon Sale Report.csv")
OUTPUT_DIR  = "data"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "asr.csv")

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(
        f"Khong tim thay file goc: {INPUT_FILE}\n"
        "Dat 'Amazon Sale Report.csv' vao thu muc Lab03/ va chay notebook tu repo root."
    )

os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(INPUT_FILE, low_memory=False)
input_rows = len(df)


In [ ]:
# ============================================================
# 1. Chuan hoa ship-state: uppercase + trim
# ============================================================
df["ship-state"] = df["ship-state"].astype("string").str.strip().str.upper()

# 2. Map alias / cach viet sai ve ten bang chuan
state_mapping = {
    "NEW DELHI": "DELHI",
    "ORISSA": "ODISHA",
    "PONDICHERRY": "PUDUCHERRY",
    "PB": "PUNJAB",
    "PUNJAB/MOHALI/ZIRAKPUR": "PUNJAB",
    "RJ": "RAJASTHAN",
    "RAJSTHAN": "RAJASTHAN",
    "RAJSHTHAN": "RAJASTHAN",
    "RAJSHSTHAN": "RAJASTHAN",
}
df["ship-state"] = df["ship-state"].replace(state_mapping)

# Sua theo thanh pho (ma bang khong ro nghia)
df.loc[(df["ship-state"] == "AR") & (df["ship-city"] == "ITANAGAR"), "ship-state"] = "ARUNACHAL PRADESH"
df.loc[(df["ship-state"] == "NL") & (df["ship-city"] == "DIMAPUR"), "ship-state"] = "NAGALAND"


In [ ]:
# ============================================================
# 3. Allow-list 36 bang chuan cua An Do (co trong dataset).
#    Sau mapping, chi giu state thuoc allow-list.
#    KHONG doan APO (khong phai bang hop le) sang bat ky bang nao.
# ============================================================
CANONICAL_STATES = {
    "ANDAMAN & NICOBAR", "ANDHRA PRADESH", "ARUNACHAL PRADESH", "ASSAM", "BIHAR",
    "CHANDIGARH", "CHHATTISGARH", "DADRA AND NAGAR", "DELHI", "GOA", "GUJARAT",
    "HARYANA", "HIMACHAL PRADESH", "JAMMU & KASHMIR", "JHARKHAND", "KARNATAKA",
    "KERALA", "LADAKH", "LAKSHADWEEP", "MADHYA PRADESH", "MAHARASHTRA", "MANIPUR",
    "MEGHALAYA", "MIZORAM", "NAGALAND", "ODISHA", "PUDUCHERRY", "PUNJAB",
    "RAJASTHAN", "SIKKIM", "TAMIL NADU", "TELANGANA", "TRIPURA", "UTTAR PRADESH",
    "UTTARAKHAND", "WEST BENGAL",
}

missing_mask = df["ship-state"].isna() | (df["ship-state"].str.strip() == "")
dropped_missing = int(missing_mask.sum())

after_missing = df[~missing_mask].copy()
invalid_mask = ~after_missing["ship-state"].isin(CANONICAL_STATES)
dropped_invalid = int(invalid_mask.sum())
unknown_values = sorted(after_missing.loc[invalid_mask, "ship-state"].unique().tolist())

df_clean = after_missing[~invalid_mask].copy()
output_rows = len(df_clean)


In [ ]:
# ============================================================
# 4. Audit summary
# ============================================================
print("=== CLEANING AUDIT ===")
print(f"input rows                : {input_rows}")
print(f"dropped (missing state)   : {dropped_missing}")
print(f"dropped (invalid state)   : {dropped_invalid}  values={unknown_values}")
print(f"output rows               : {output_rows}")
print(f"output unique states      : {df_clean['ship-state'].nunique()}")

# Assert: khong con state null/empty/unknown
assert df_clean["ship-state"].notna().all(), "Con null ship-state"
assert (df_clean["ship-state"].str.strip() != "").all(), "Con empty ship-state"
assert set(df_clean["ship-state"].unique()).issubset(CANONICAL_STATES), "Con unknown state"

# 5. Xuat canonical input. Giu nguyen cot/thu tu cot/index goc.
df_clean.to_csv(OUTPUT_FILE, index=False)
print(f"\nWrote {OUTPUT_FILE}  ({output_rows} rows, {df_clean['ship-state'].nunique()} states)")
